# SpriteForge trên Kaggle: mô tả nhân vật → sprite sheet pixel art

Mỗi frame được vẽ bằng **SD1.5 pixel art + ControlNet OpenPose** theo thư viện tư thế (đi, chạy, chém, nhảy...) và **IP-Adapter** để giữ ngoại hình nhân vật; sau đó tách nền, ép palette, đóng gói sheet. (Engine video WAN 2.2 vẫn còn, bật bằng `"engine": "video"`.)

**Chuẩn bị (một lần, chi tiết trong `kaggle/README.md`):**
1. Kaggle → Settings: **verify số điện thoại** (bắt buộc để bật GPU và Internet).
2. Panel bên phải → *Session options*: **Accelerator = GPU T4 x2** (2 GPU → nhanh gấp đôi), **Internet = On**.
3. **Add Input** → dataset `spriteforge-models` (checkpoint SD1.5 pixel art) **và** output của notebook `SpriteForge Models` (ControlNet, IP-Adapter; xem `kaggle/SpriteForge_Models.ipynb`). Thiếu model nào thì cell 2 tự tải từ HuggingFace.
4. Code SpriteForge: clone từ GitHub theo `SPRITEFORGE_REPO` + `SPRITEFORGE_BRANCH` ở cell 1. Repo private thì thêm secret `GITHUB_TOKEN` (*Add-ons → Secrets*).

**Chạy:** sửa nội dung `jobs.json` ở cell 5 → **Run All**, hoặc *Save Version → Save & Run All (Commit)* để chạy nền, không cần mở trình duyệt.
Kết quả: `output.zip` trong panel **Output**.

In [ ]:
# ==== 1. CẤU HÌNH ====
SPRITEFORGE_REPO = "https://github.com/neiknart2033/Text-To-Sprite-Sheet"  # bỏ qua nếu có dataset spriteforge-code
SPRITEFORGE_BRANCH = "v1"
# Repo private: Add-ons → Secrets → thêm GITHUB_TOKEN (fine-grained token, Contents: Read-only) và bật cho notebook này
COMFY_REF = ""           # để trống = ComfyUI mới nhất; điền tag/commit nếu muốn cố định phiên bản
RUN_BENCHMARK = False    # True = chạy thử 1 action (cell 4) để đo tốc độ WAN trên GPU này
NUM_GPUS = "auto"        # "auto" = dùng mọi GPU (T4 x2 -> 2 ComfyUI song song); đặt 1 nếu báo thiếu RAM
DOWNLOAD_WAN = False     # True nếu có action "engine": "video" mà chưa có file WAN (~17GB)

WORK = "/tmp/spriteforge"            # ComfyUI + model (không lẫn vào Output)
OUTPUT = "/kaggle/working/output"    # kết quả (tải về từ panel Output)

In [ ]:
# ==== 2. CÀI ĐẶT: code SpriteForge + ComfyUI + model (~5–10 phút, 1 lần mỗi session) ====
import base64, glob, os, shutil, subprocess, sys

if "WORK" not in globals():
    raise RuntimeError("Chưa chạy cell 1 (CẤU HÌNH). Mỗi lần session khởi động lại "
                       "(VD sau khi đổi GPU/Internet) phải chạy lại từ cell 1.")

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def git(*args, cwd=None):
    """git có token GITHUB_TOKEN (nếu có secret) qua header, không lộ token ra output."""
    auth = []
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        auth = ["-c", "http.extraHeader=Authorization: Basic "
                + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
    except Exception:
        pass  # không có secret -> repo phải public
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode:
        raise RuntimeError(f"git {args[0]} lỗi:\n{r.stderr.strip()}\n"
                           "→ Repo private: thêm secret GITHUB_TOKEN. Sai nhánh: sửa SPRITEFORGE_BRANCH.")
    return r.stdout.strip()

os.makedirs(WORK, exist_ok=True)

# --- code SpriteForge: dataset "spriteforge-code" nếu có, không thì clone/pull từ GitHub
APP = f"{WORK}/app"
found = glob.glob("/kaggle/input/**/forge/pipelines.py", recursive=True)
if found:
    if not os.path.exists(APP):
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), APP)
    print("Code lấy từ dataset:", os.path.dirname(os.path.dirname(found[0])))
else:
    if os.path.exists(f"{APP}/.git"):
        git("pull", "--ff-only", cwd=APP)  # chạy lại cell = lấy code mới nhất
    else:
        git("clone", "--depth", "1", "--branch", SPRITEFORGE_BRANCH, SPRITEFORGE_REPO, APP)
    print("Code GitHub:", git("log", "-1", "--format=%h %s (%cr)", cwd=APP))

# --- ComfyUI
COMFY = f"{WORK}/ComfyUI"
if not os.path.exists(COMFY):
    if COMFY_REF:
        sh(f"git clone https://github.com/comfyanonymous/ComfyUI {COMFY} && cd {COMFY} && git checkout {COMFY_REF}")
    else:
        sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
sh(f"pip install -q -r {COMFY}/requirements.txt")

# --- custom node IP-Adapter (giữ ngoại hình nhân vật giữa các frame)
IPA = f"{COMFY}/custom_nodes/ComfyUI_IPAdapter_plus"
if not os.path.exists(IPA):
    sh(f"git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus {IPA}")

# --- model: tìm trong Input (dataset/notebook) theo tên file, symlink; thiếu thì tải từ HuggingFace
WAN_REPO = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
MODEL_FILES = [  # (thư mục ComfyUI, repo HuggingFace, đường dẫn trong repo, tên lưu, cần tải?)
    ("controlnet", "comfyanonymous/ControlNet-v1-1_fp16_safetensors",
     "control_v11p_sd15_openpose_fp16.safetensors", "control_v11p_sd15_openpose_fp16.safetensors", True),
    ("ipadapter", "h94/IP-Adapter", "models/ip-adapter-plus_sd15.safetensors",
     "ip-adapter-plus_sd15.safetensors", True),
    ("clip_vision", "h94/IP-Adapter", "models/image_encoder/model.safetensors",
     "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors", True),
    ("diffusion_models", WAN_REPO, "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
     "wan2.2_ti2v_5B_fp16.safetensors", DOWNLOAD_WAN),
    ("text_encoders", WAN_REPO, "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "umt5_xxl_fp8_e4m3fn_scaled.safetensors", DOWNLOAD_WAN),
    ("vae", WAN_REPO, "split_files/vae/wan2.2_vae.safetensors", "wan2.2_vae.safetensors", DOWNLOAD_WAN),
]
MODELS = f"{WORK}/models"
for folder in ("checkpoints", *(m[0] for m in MODEL_FILES)):
    os.makedirs(f"{MODELS}/{folder}", exist_ok=True)

known = {m[3] for m in MODEL_FILES}
inputs = glob.glob("/kaggle/input/**/*.safetensors", recursive=True) + glob.glob("/kaggle/input/**/*.ckpt", recursive=True)
for f in inputs:  # mọi file .safetensors/.ckpt KHÁC các model trên -> coi là checkpoint SD1.5
    name = os.path.basename(f)
    dst = f"{MODELS}/checkpoints/{name}"
    if name not in known and not os.path.exists(dst):
        os.symlink(f, dst)

for folder, repo, path, name, needed in MODEL_FILES:
    dst = f"{MODELS}/{folder}/{name}"
    if os.path.exists(dst):
        continue
    in_input = [f for f in inputs if os.path.basename(f) == name]
    if in_input:
        os.symlink(in_input[0], dst)
        print("Model từ Input:", name)
    elif needed:
        from huggingface_hub import hf_hub_download
        print("Tải từ HuggingFace:", name, "...")
        shutil.move(hf_hub_download(repo, path, local_dir=f"{WORK}/hf"), dst)

# IP-Adapter tìm model trong ComfyUI/models/ipadapter: symlink thêm vào đó cho chắc
os.makedirs(f"{COMFY}/models/ipadapter", exist_ok=True)
for name in os.listdir(f"{MODELS}/ipadapter"):
    if not os.path.exists(f"{COMFY}/models/ipadapter/{name}"):
        os.symlink(f"{MODELS}/ipadapter/{name}", f"{COMFY}/models/ipadapter/{name}")

with open(f"{COMFY}/extra_model_paths.yaml", "w") as f:
    f.write(f"spriteforge:\n  base_path: {MODELS}\n" + "".join(
        f"  {folder}: {folder}\n" for folder in sorted(os.listdir(MODELS))))

for folder in sorted(os.listdir(MODELS)):
    print(f"{folder:17s}", os.listdir(f"{MODELS}/{folder}"))
if not os.listdir(f"{MODELS}/checkpoints"):
    print("CHÚ Ý: chưa có checkpoint SD1.5 -> chỉ chạy được job mode \"image\" (sprite có sẵn). "
          "Muốn tạo nhân vật từ mô tả: Add Input dataset chứa checkpoint.")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# ==== 3. KHỞI ĐỘNG ComfyUI: mỗi GPU một tiến trình chạy nền (port 8188, 8189, ...) ====
import time, urllib.request
assert "COMFY" in globals(), "Chạy cell 1 và cell 2 trước"

smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
gpu_ids = [l.split(":")[0].split()[-1] for l in smi.splitlines() if l.startswith("GPU")]
n_gpus = len(gpu_ids) if NUM_GPUS == "auto" else min(int(NUM_GPUS), len(gpu_ids))
assert n_gpus > 0, "Không thấy GPU: Session options → Accelerator = GPU T4 x2"

def comfy_up(port):
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{port}/system_stats", timeout=3).status == 200
    except Exception:
        return False

def start_comfy(i):
    port, d, log_path = 8188 + i, f"{WORK}/gpu{i}", f"{WORK}/comfy{i}.log"
    base = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(port), "--preview-method", "none"]
    # Thư mục riêng cho từng tiến trình (ảnh tải lên, file tạm) để 2 ComfyUI không giẫm nhau
    dirs = ["--input-directory", f"{d}/input", "--output-directory", f"{d}/output",
            "--temp-directory", d, "--user-directory", f"{d}/user"]
    for sub in ("input", "output", "user"):  # --user-directory đòi thư mục có sẵn
        os.makedirs(f"{d}/{sub}", exist_ok=True)
    # PYTHONUNBUFFERED: log ghi ra file ngay, không kẹt trong bộ đệm (lỗi mới đọc được)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu_ids[i], "PYTHONUNBUFFERED": "1"}
    for args in (base + dirs, base):  # ComfyUI từ chối cờ thư mục -> chạy lại không cờ
        with open(log_path, "w") as log_file:
            proc = subprocess.Popen(args, cwd=COMFY, stdout=log_file, stderr=subprocess.STDOUT, env=env)
        for _ in range(300):  # tối đa ~10 phút
            if comfy_up(port) or proc.poll() is not None:
                break
            time.sleep(2)
        if comfy_up(port):
            return port
        text = open(log_path).read()
        if not ("usage:" in text and "error:" in text):  # không phải lỗi tham số -> thử lại vô ích
            break
    state = "đang treo (vẫn chạy)" if proc.poll() is None else f"đã thoát, mã {proc.returncode}"
    if proc.poll() is None:
        proc.kill()
    print(f"----- {log_path}: tiến trình {state} -----")
    print(open(log_path).read()[-4000:] or "(log trống)")
    raise RuntimeError(f"ComfyUI trên GPU {i} không khởi động được ({state}), xem log ở trên")

# Luôn khởi động lại: ComfyUI chỉ đọc thư mục model (extra_model_paths.yaml) và custom node
# lúc khởi động. Dùng lại tiến trình cũ (chạy từ trước cell 2) -> "không thấy model/node".
if subprocess.run(["pkill", "-f", "main.py --listen 127.0.0.1"]).returncode == 0:
    print("Tắt ComfyUI cũ, khởi động lại...")
    for _ in range(30):
        if not any(comfy_up(8188 + i) for i in range(len(gpu_ids))):
            break
        time.sleep(1)

ports = []
for i in range(n_gpus):
    try:
        ports.append(start_comfy(i))
    except RuntimeError as e:
        if not ports:
            raise
        print(f"CẢNH BÁO: {e}. Chạy tiếp với {len(ports)} GPU (gửi log trên cho người bảo trì).")
COMFY_URLS = ",".join(f"http://127.0.0.1:{p}" for p in ports)
print(f"ComfyUI sẵn sàng trên {len(ports)} GPU:", COMFY_URLS)

In [ ]:
# ==== 4. (TÙY CHỌN) BENCHMARK: 1 nhân vật + 1 action, in thời gian từng bước ====
assert "COMFY_URLS" in globals(), "Chạy cell 1, 2, 3 trước"
if RUN_BENCHMARK:
    !python {APP}/kaggle/run_batch.py --jobs {APP}/kaggle/jobs.bench.json --out /kaggle/working/bench --comfy-url {COMFY_URLS}

In [ ]:
%%writefile /kaggle/working/jobs.json
{
  "comfy_url": "http://127.0.0.1:8188",
  "checkpoint": "auto",
  "seed": -1,
  "steps": 30,
  "cfg_scale": 7,
  "frame_size": 128,
  "char_height": 64,
  "palette": 32,
  "fps": 10,
  "view": "side",
  "bg_color": "auto",
  "preview_scale": 3,
  "wan": {
    "steps": 20,
    "duration": 2.0
  },
  "jobs": [
    {
      "name": "hac_ronin",
      "mode": "prompt",
      "prompt": "dark samurai in black-indigo armor, oni mask with glowing red eyes, tattered black cloak, two katanas at the waist",
      "character_image": null,
      "actions": [
        {"name": "idle",   "prompt": "standing still in a ready stance, calm breathing, cloak moving slightly", "frames": 8},
        {"name": "walk",   "prompt": "walking in place, steady walk cycle, legs and arms swinging", "frames": 8, "align": "track"},
        {"name": "run",    "prompt": "running in place, fast run cycle, leaning forward", "frames": 8, "align": "track"},
        {"name": "attack", "prompt": "draws a katana and slashes horizontally, then returns to stance", "frames": 8},
        {"name": "heavy",  "prompt": "raises the katana overhead and strikes down hard", "frames": 8},
        {"name": "hurt",   "prompt": "recoils backward from a hit, then recovers", "frames": 6, "loop": false},
        {"name": "jump",   "prompt": "crouches, jumps straight up, then lands", "frames": 6, "loop": false},
        {"name": "death",  "prompt": "staggers, drops to the knees and collapses to the ground", "frames": 8, "loop": false, "duration": 3.0}
      ]
    }
  ]
}

In [ ]:
# ==== 6. CHẠY BATCH (chạy lại cell này = tiếp tục từ action chưa xong) ====
assert "COMFY_URLS" in globals(), "Chạy cell 1, 2, 3 trước"
!python {APP}/kaggle/run_batch.py --jobs /kaggle/working/jobs.json --out {OUTPUT} --comfy-url {COMFY_URLS}

In [ ]:
# ==== 7. ĐÓNG GÓI: bấm link "output.zip" hiện ra bên dưới để tải về ====
# (Kaggle không cho tải cả thư mục, chỉ tải từng file -> gom thành 1 file zip)
!cd /kaggle/working && rm -f output.zip && zip -qr output.zip output && ls -lh output.zip
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink("output.zip", result_html_prefix="Tải kết quả: "))

In [ ]:
# ==== 8. (TÙY CHỌN) XEM NHANH các preview.gif ngay trong notebook ====
from IPython.display import HTML, display
gifs = sorted(glob.glob(f"{OUTPUT}/*/*/preview.gif"))
display(HTML("".join(
    f'<figure style="display:inline-block;margin:6px;text-align:center">'
    f'<img src="{os.path.relpath(g, "/kaggle/working")}" style="image-rendering:pixelated;background:#444">'
    f'<figcaption>{os.path.relpath(os.path.dirname(g), OUTPUT)}</figcaption></figure>' for g in gifs)))